In [1]:
!pip install -q --upgrade openai python-dotenv

In [2]:
!pip install gradio

  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.7 MB 4.8 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.7 MB 5.9 MB/s eta 0:00:04
   ------ --------------------------------- 3.1/19.7 MB 6.4 MB/s eta 0:00:03
   --------- ------------------------------ 4.7/19.7 MB 6.1 MB/s eta 0:00:03
   ------------ --------------------------- 6.0/19.7 MB 6.3 MB/s eta 0:00:03
   -------------- ------------------------- 7.3/19.7 MB 6.3 MB/s eta 0:00:02
   ----------------- ---------------------- 8.7/19.7 MB 6.2 MB/s eta 0:00:02
   -------------------- ------------------- 10.2/19.7 MB 6.3 MB/s eta 0:00:02
   ----------------------- ---------------- 11.5/19.7 MB 6.4 MB/s eta 0:00:02
   -------------------------- ------------- 12.8/19.7 MB 6.4 MB/s eta 0:00:02
   ---------------------------- ----------- 14.2/19.7 MB 6.4 MB/s eta 0:00:01
   ------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [1]:
# Import necessary libraries
import os
from IPython.display import display, Markdown
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Retrieve the OpenAI API key from environment variables
LM_STUDIO_BASE_URL = os.getenv("LM_STUDIO_BASE_URL")

print("OpenAI API Key loaded successfully.")
# Let's view the first few characters to confirm it's loaded (DO NOT print the full key)
print(f"Key starts with: {LM_STUDIO_BASE_URL[:10]}...")

# Configure the OpenAI Client using the loaded key
openai_client = OpenAI(
    base_url= LM_STUDIO_BASE_URL,
    api_key ="lm-studio"
)
print("LM Studio client successfully configured.")


OpenAI API Key loaded successfully.
Key starts with: http://127...
LM Studio client successfully configured.


In [2]:
# Define a helper function to display markdown nicely
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

In [3]:
# Let's define the Python function to get a response from the AI Tutor
def get_ai_tutor_response(user_question,selected_model):
    """
    Sends a question to the OpenAI API, asking it to respond as an AI Tutor.

    Args:
        user_question (str): The question asked by the user.

    Returns:
        str: The AI's response, or an error message.
    """
    # Define the system prompt - instructions for the AI's personality and role
    system_prompt = "You are a helpful and patient AI Tutor. Explain concepts clearly and concisely."

    try:
        # Make the API call to OpenAI
        response = openai_client.chat.completions.create(
            model = selected_model,  # A fast and capable model suitable for tutoring
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_question}],
            temperature = 0.7,  # Allows for some creativity but keeps responses focused
        )
        # Extract the answer content
        ai_response = response.choices[0].message.content
        return ai_response

    except Exception as e:
        # Handle potential errors during the API call
        print(f"An error occurred: {e}")
        return f"Sorry, I encountered an error trying to get an answer: {e}"

In [33]:
# Function to get available models from LM Studio
def get_models():
    models = openai_client.models.list()
    return [model.id for model in models.data]

In [34]:
# Fetch models from LM Studio
available_models = get_models()


In [14]:
# Let's test our function with a sample question
test_question = "Could you explain the concepts in React and their purpose in programming?"
print_markdown(f"Asking the AI Tutor: '{test_question}'")

# Call the function and store the response
tutor_answer = get_ai_tutor_response(test_question,selected_model)

# Print the AI's response
print_markdown("\n🤖 AI Tutor's Response:\n")
print_markdown(tutor_answer)

Asking the AI Tutor: 'Could you explain the concepts in React and their purpose in programming?'

An error occurred: Error code: 400 - {'error': 'Model unloaded.'}



🤖 AI Tutor's Response:


Sorry, I encountered an error trying to get an answer: Error code: 400 - {'error': 'Model unloaded.'}

In [15]:
import gradio as gr

In [35]:
# Define the mapping for explanation levels
explanation_levels = {
    1: "student",
    2: "junior",
    3: "senior",
    4: "Expert",
}

In [36]:
# Let's create a new function that streams the response
def stream_ai_tutor_response(user_question,selected_model,explanation_level_value):
    """
    Sends a question to the OpenAI API and streams the response as a generator.

    Args:
        user_question (str): The question asked by the user.

    Yields:
        str: Chunks of the AI's response.
    """

    # Get the descriptive text for the chosen level
    level_description = explanation_levels.get(
        explanation_level_value, "clearly and concisely"
    )  # Default if level not found
    
    system_prompt = f"You are a helpful and patient AI Tutor. Explain concepts {level_description}."

    print(f"DEBUG: Using System Prompt: '{system_prompt}'")  # For checking
    
    try:
        # Note: stream = True is the key change here!
        stream = openai_client.chat.completions.create(
            model = selected_model,
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_question}],
            temperature = 0.7,
            stream = True,  # Enable streaming (magic happens here)
        )

        # Iterate through the response chunks
        full_response = ""  # Keep track of the full response if needed later

        # Loop through each chunk of the response as it arrives
        for chunk in stream:
            # Check if this chunk contains actual text content
            if chunk.choices[0].delta and chunk.choices[0].delta.content:
                # Extract the text from this chunk
                text_chunk = chunk.choices[0].delta.content
                # Add this chunk to our growing response
                full_response += text_chunk
                # 'yield' is special - it sends the current state of the response to Gradio
                # This makes the text appear to be typing in real-time
                yield full_response

    except Exception as e:
        print(f"An error occurred during streaming: {e}")
        yield f"Sorry, I encountered an error: {e}"

In [43]:
# Let's define the Gradio interface
# fn: The function to wrap (our AI tutor function)
# inputs: A component for the user to type their question
# outputs: A component to display the AI's answer
# title/description: Text for the UI heading
ai_tutor_interface_simple = gr.Interface(
    fn = stream_ai_tutor_response,
    inputs = [gr.Textbox(lines = 2, placeholder = "Ask the AI Tutor anything...", label = "Your Question"),
              gr.Dropdown(
            choices=available_models,
            value=available_models[0] if available_models else None,
            label="Choose Model"
        ),       gr.Slider(
            minimum = 1,
            maximum = 4,
            step = 1,  # Only allow whole numbers
            value = 3,  # Default level (high school)
            label = "Explanation Level",  # Label for the slider
        ),],
    outputs = gr.Markdown(label = "AI Tutor's Explanation (Streaming)", container = True, height = 250),
    title = "🤖 AI Tutor",
    description = "Enter your question below and the AI Tutor will provide an explanation. Powered by LM Studio API.",
    flagging_mode = "never",  # Disables the flagging feature for simplicity
)

# Launch the interface!
# This will typically create a link (or display inline in environments like Google Colab/Jupyter)
# You can interact with this UI directly.
print("Launching Gradio Interface...")
ai_tutor_interface_simple.launch()

Launching Gradio Interface...
* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


DEBUG: Using System Prompt: 'You are a helpful and patient AI Tutor. Explain concepts Expert.'
An error occurred during streaming: Context size has been exceeded.
